# Frontend V2 — pinned Qwen3-4B QLoRA

**Build status: BLOCKED.** Training/runtime compatibility remains provisional.
UPLOADABLE PREVIEW — NOT TRAINABLE. Zero approved data are embedded. Do not select a GPU or Run all yet.
Blocked inputs (local contract):
- missing required V2 input: datasets/frontend-stack/v2/final/train.jsonl
- missing required V2 input: datasets/frontend-stack/v2/final/holdout.jsonl
- missing required V2 input: datasets/frontend-stack/v2/final/train-short3072.jsonl
- missing required V2 input: datasets/frontend-stack/v2/final/holdout-short3072.jsonl
- missing required V2 input: datasets/frontend-stack/v2/final/manifest.json

To prepare a trainable export **on your own computer**, supply reviewed final V2 inputs
(180 approved gold and 60 separately reviewed general rows), then run
`py -3 scripts/v2_training_contract.py` and `py -3 scripts/build_v2_colab.py`.
The first command must exit 0, and the regenerated notebook must embed a nonempty
hash-bound payload. Upload `notebooks/frontend_stack_qwen3_4b_colab.ipynb` via
Colab **File → Upload notebook**. Only then choose a T4/L4/A100 GPU and Run all.
No V1 data fallback. Never edit the status flag to bypass a gate. The first code
cell rechecks the full restored contract before pip, torch or weight downloads.

Recipe: 3,072 whole-record tokens, rank 16, assistant-only labels (including tool
calls), two epochs, separate non-skipped/nonzero longest-training-record update
plus longest-holdout memory/finite-loss probe, clean-base reload, final holdout loss,
content-bound checkpoints, full Q4_K_M tensor/tokenizer/template validation.
No implicit 2,048 fallback: rebuild/review/re-pin derivatives first.
Package pins come from the archived repository recipe; 4B compatibility has not
been tested. A real clean Colab GPU optimizer probe is mandatory, not a waiver.
Export is provisional until independent serving smoke and frozen A/B gates pass.
Spot-check approval must use the declared human or independent-AI policy, never an invented human. File hashes cannot establish reviewer identity.


In [ ]:
# Model-free gate FIRST. Executing later cells out of order also fails closed.
import base64, hashlib, io, json, platform, shutil, sys, zipfile
from pathlib import Path, PurePosixPath
BUILD_REPORT = json.loads("{\"artifact_hashes\":{},\"blockers\":[\"missing required V2 input: datasets/frontend-stack/v2/final/train.jsonl\",\"missing required V2 input: datasets/frontend-stack/v2/final/holdout.jsonl\",\"missing required V2 input: datasets/frontend-stack/v2/final/train-short3072.jsonl\",\"missing required V2 input: datasets/frontend-stack/v2/final/holdout-short3072.jsonl\",\"missing required V2 input: datasets/frontend-stack/v2/final/manifest.json\"],\"counts\":{},\"data_ready\":false,\"runtime_compatibility\":\"unverified\",\"schema_version\":1,\"specialist\":\"frontend-stack\",\"status\":\"BLOCKED\",\"train_ready\":false}")
PAYLOAD_B64 = ''
PAYLOAD_HASHES = {}
READY = False
if not BUILD_REPORT['data_ready'] or not PAYLOAD_B64:
    print('NOT TRAINABLE: regenerate locally after the approved V2 corpus passes its contract.')
    raise RuntimeError('BLOCKED: ' + json.dumps(BUILD_REPORT['blockers']))
if platform.system() != 'Linux' or not Path('/content').is_dir():
    raise RuntimeError('Colab GPU only; no local training')
from google.colab import files
if shutil.disk_usage('/content').free < 25 * 1024**3:
    raise RuntimeError('Need at least 25 GiB free disk')
WORK = Path('/content/Master-Models-V2')
raw = base64.b64decode(PAYLOAD_B64, validate=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    if len(archive.namelist()) != len(PAYLOAD_HASHES) or set(archive.namelist()) != set(PAYLOAD_HASHES):
        raise RuntimeError('Payload membership mismatch')
    for name in archive.namelist():
        parts = PurePosixPath(name)
        if parts.is_absolute() or '..' in parts.parts or '\\' in name or ':' in name:
            raise RuntimeError('Unsafe payload path')
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != PAYLOAD_HASHES[name]:
            raise RuntimeError('Payload hash mismatch: ' + name)
        target = WORK / name
        if not target.resolve().is_relative_to(WORK.resolve()): raise RuntimeError('Payload escapes root')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
sys.path.insert(0, str(WORK / 'scripts'))
from v2_training_contract import check_contract
REPORT = check_contract(WORK)
if not REPORT['data_ready'] or REPORT['contract_sha256'] != BUILD_REPORT['contract_sha256']:
    raise RuntimeError('Restored contract failed: ' + json.dumps(REPORT))
READY = True
print('Data validated. Runtime readiness still unverified.')


In [ ]:
if not globals().get('READY', False): raise RuntimeError('BLOCKED before pip')
if not check_contract(WORK)['data_ready']: raise RuntimeError('Inputs changed before pip')
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r',
                str(WORK / 'training/requirements-v2-colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
print('Packages installed; compatibility is not established until the real probe passes.')


In [ ]:
if not globals().get('READY', False): raise RuntimeError('BLOCKED before runtime setup')
RUNTIME_SOURCE = "import argparse, hashlib, importlib.metadata, importlib.util, json, os, platform, shutil, struct, sys\nfrom pathlib import Path\nsys.path.insert(0, str(Path(__file__).resolve().parent / 'scripts'))\nfrom v2_training_contract import (check_contract, load_json, load_rows, canonical, sha256,\n    require, assistant_labels, BASE, REVISION, CANONICAL, CANONICAL_REVISION, TOKENIZER_FILES,\n    validate_probe_evidence, checkpoint_hashes, validate_checkpoint, validate_gguf_reader, file_sha256,\n    discover_gguf_exports)\n\n\ndef gguf_type(path):\n    # Parse GGUF v3 metadata, not the filename. GGML_FTYPE_MOSTLY_Q4_K_M = 15.\n    with path.open('rb') as stream:\n        def read(fmt):\n            size = struct.calcsize(fmt)\n            data = stream.read(size)\n            require(len(data) == size, 'truncated GGUF')\n            return struct.unpack(fmt, data)[0]\n        def string():\n            size = read('<Q')\n            require(size <= 2**24, 'unreasonable GGUF string')\n            data = stream.read(size)\n            require(len(data) == size, 'truncated GGUF string')\n            return data.decode('utf-8')\n        def value(kind, depth=0):\n            require(depth < 4, 'invalid GGUF nesting')\n            formats = {0:'<B', 1:'<b', 2:'<H', 3:'<h', 4:'<I', 5:'<i', 6:'<f', 7:'<?', 10:'<Q', 11:'<q', 12:'<d'}\n            if kind in formats: return read(formats[kind])\n            if kind == 8: return string()\n            if kind == 9:\n                subtype, count = read('<I'), read('<Q')\n                require(count <= 10**7, 'invalid GGUF array length')\n                for _ in range(count): value(subtype, depth + 1)\n                return None\n            raise RuntimeError('unknown GGUF metadata type')\n        require(stream.read(4) == b'GGUF' and read('<I') == 3, 'not GGUF v3')\n        tensors, count = read('<Q'), read('<Q')\n        require(tensors > 0 and count < 10**6, 'invalid GGUF counts')\n        file_type = None\n        for _ in range(count):\n            key, kind = string(), read('<I')\n            result = value(kind)\n            if key == 'general.file_type': file_type = result\n        require(file_type == 15, 'export is not Q4_K_M (general.file_type != 15)')\n        return file_type\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--mode', choices=('probe', 'train'), required=True)\n    ap.add_argument('--resume')\n    args = ap.parse_args()\n    resume_from_checkpoint = None\n    root = Path(__file__).resolve().parent\n    report = check_contract(root)\n    require(report['data_ready'], 'BLOCKED before ML imports: ' + canonical(report['blockers']))\n    require(platform.system() == 'Linux' and Path('/content').is_dir()\n            and importlib.util.find_spec('google.colab') is not None, 'Colab only; local GPU training prohibited')\n    require(shutil.disk_usage(root).free >= 25 * 1024**3, 'need 25 GiB free disk before base download')\n    os.environ['UNSLOTH_CE_LOSS_TARGET_GB'] = '0.05'\n    os.environ['UNSLOTH_DISABLE_DOUBLE_BUFFER'] = '1'\n    # Unsloth patch ordering is significant.\n    import unsloth\n    import torch\n    from unsloth import FastLanguageModel, is_bfloat16_supported\n    from datasets import Dataset\n    from transformers import AutoTokenizer, TrainerCallback\n    from huggingface_hub import snapshot_download\n    from trl import SFTTrainer, SFTConfig\n    from importlib.metadata import version\n    requirements = (root / 'training/requirements-v2-colab.txt').read_text().splitlines()\n    expected = dict(line.strip().split('==') for line in requirements if line.strip() and not line.startswith('#'))\n    packages = {name: version(name) for name in expected}\n    require(packages == expected, 'installed direct package pins differ')\n    require(torch.cuda.is_available(), 'CUDA unavailable')\n    gpu = torch.cuda.get_device_properties(0)\n    free, total = torch.cuda.mem_get_info()\n    require(any(name in gpu.name for name in ('T4', 'L4', 'A100')) and total >= 14.5 * 1024**3\n            and free >= 13 * 1024**3, 'need clean T4/L4/A100 with >=13 GiB free VRAM')\n    runtime = {'python': platform.python_version(), 'packages': packages,\n               'torch': torch.__version__, 'cuda': torch.version.cuda,\n               'triton': version('triton'), 'gpu': gpu.name, 'vram_bytes': total,\n               'installed_distributions':dict(sorted((dist.metadata['Name'],dist.version)\n                   for dist in importlib.metadata.distributions() if dist.metadata['Name']))}\n    params = {'rank':16, 'alpha':32, 'dropout':0, 'max_seq_length':3072,\n              'epochs':2, 'batch':1, 'accumulation':16, 'lr':2e-4,\n              'seed':3407, 'optimizer':'paged_adamw_8bit', 'assistant_only':True,\n              'packing':False, 'base':BASE, 'revision':REVISION,\n              'targets':['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']}\n    run_contract = {'data_contract': report['contract_sha256'], 'runtime':runtime, 'parameters':params,\n                    'runtime_program_sha256':sha256(Path(__file__).read_bytes()),\n                    'helper_sha256':{name:sha256((root / 'scripts' / name).read_bytes())\n                                     for name in ('v2_training_contract.py','build_short_train.py')}}\n    run_hash = sha256(canonical(run_contract).encode())\n    out = root / 'outputs'\n    out.mkdir(exist_ok=True)\n    contract_file = out / 'run-contract.json'\n    if args.mode == 'train':\n        probe = load_json(out / 'probe.json')\n        require(probe.get('run_sha256') == run_hash and probe.get('passed') is True, 'real isolated probe missing/stale; no waiver')\n        validate_probe_evidence(probe)\n        if contract_file.exists():\n            require(load_json(contract_file) == run_contract, 'existing run contract differs')\n        contract_file.write_text(canonical(run_contract))\n        if args.resume:\n            checkpoint = Path(args.resume).resolve()\n            require(checkpoint.parent == (out / 'trainer').resolve() and checkpoint.name.startswith('checkpoint-'), 'invalid resume path')\n            validate_checkpoint(checkpoint, run_contract)\n            resume_from_checkpoint = str(checkpoint)\n        else:\n            require(not list((out / 'trainer').glob('checkpoint-*')) and not (out / 'holdout-evaluation.json').exists(), 'existing run requires explicit checkpoint resume or a fresh runtime')\n    else:\n        require(not args.resume, 'probe cannot resume')\n        (out / 'probe.json').unlink(missing_ok=True)\n\n    # Download/verify only tokenizer metadata first; base weights come later.\n    pin = load_json(root / 'training/templates/qwen3-4b.pin.json')\n    base_pin = load_json(root / 'training/templates/qwen3-4b.base-pin.json')\n    native_path = Path(snapshot_download(BASE, revision=REVISION,\n                       allow_patterns=list(TOKENIZER_FILES) + ['config.json', 'special_tokens_map.json']))\n    canonical_path = Path(snapshot_download(CANONICAL, revision=CANONICAL_REVISION,\n                          allow_patterns=list(TOKENIZER_FILES) + ['config.json', 'special_tokens_map.json']))\n    canonical_hashes = dict(pin['tokenizer_files'], **{'merges.txt':pin['merges_sha256']})\n    for folder, hashes in ((native_path, base_pin['native_tokenizer_files']), (canonical_path, canonical_hashes)):\n        for name, digest in hashes.items():\n            require(sha256((folder / name).read_bytes()) == digest, 'tokenizer file drift: ' + name)\n    model_config = load_json(native_path / 'config.json')\n    native_config = load_json(native_path / 'tokenizer_config.json')\n    require(sha256(native_config['chat_template'].encode()) == base_pin['native_template_sha256'], 'native template drift before normalization')\n    native = AutoTokenizer.from_pretrained(str(native_path), local_files_only=True)\n    canonical_tokenizer = AutoTokenizer.from_pretrained(str(canonical_path), local_files_only=True)\n    native_pad = native.pad_token\n    normalization = base_pin['normalization']\n    require(native_pad == normalization['native_pad_token'] and native.pad_token_id == normalization['native_pad_token_id'], 'native pad drift')\n    require(canonical_tokenizer.pad_token == normalization['canonical_pad_token']\n            and canonical_tokenizer.pad_token_id == normalization['canonical_pad_token_id'], 'canonical pad drift')\n    template = (root / 'training/templates/qwen3-4b.jinja').read_text(encoding='utf-8')\n    require(sha256(template.encode()) == pin['template_sha256'], 'canonical template drift')\n    # Only the documented verified normalization is allowed. Do not replace pad.\n    native.chat_template = template\n    canonical_tokenizer.chat_template = template\n    prefix = root / 'datasets/frontend-stack/v2/final'\n    rows = {split:load_rows(prefix / (split + '-short3072.jsonl')) for split in ('train', 'holdout')}\n    token_data, lengths = {}, {}\n    for split, records in rows.items():\n        processed, counts = [], []\n        for row in records:\n            for msg in row['messages']:\n                require(not any(marker in canonical(msg) for marker in ('<|im_start|>', '<|im_end|>')), 'turn-marker injection in source record')\n            options = dict(tools=row.get('tools') or None, tokenize=False,\n                           add_generation_prompt=False, enable_thinking=False)\n            text = native.apply_chat_template(row['messages'], **options)\n            canonical_text = canonical_tokenizer.apply_chat_template(row['messages'], **options)\n            encoded = native(text, add_special_tokens=False, return_offsets_mapping=True)\n            ids = encoded['input_ids']\n            require(ids == canonical_tokenizer(canonical_text, add_special_tokens=False)['input_ids'], 'unpadded token parity failure')\n            require(text.strip() and 0 < len(ids) <= 3072, 'whole record over cap; no slicing/truncation/fallback')\n            require(not row.get('tools') or '<tools>' in text, 'missing tool definitions')\n            # Locate complete template assistant spans. Tool results render as user\n            # spans. Keep assistant tool-call JSON AND turn-ending tokens as labels.\n            expected_calls = sum(len(m.get('tool_calls') or []) for m in row['messages'])\n            labels = assistant_labels(text, ids, encoded['offset_mapping'], expected_calls)\n            processed.append({'input_ids':ids, 'attention_mask':[1]*len(ids), 'labels':labels})\n            counts.append(len(ids))\n        token_data[split], lengths[split] = processed, counts\n    (out / ('probe-token-bounds.json' if args.mode == 'probe' else 'token-bounds.json')).write_text(canonical({\n        split:{'records':len(values),'min':min(values),'max':max(values)} for split,values in lengths.items()}))\n\n    model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE, revision=REVISION,\n                    max_seq_length=3072, load_in_4bit=True)\n    require(tokenizer.get_vocab() == native.get_vocab(), 'loaded base tokenizer vocabulary drift')\n    require(tokenizer.pad_token == native_pad and tokenizer.pad_token_id == normalization['native_pad_token_id'], 'loader changed native pad; refuse repair')\n    # Loader-specific templates are not trusted; raw native metadata was verified\n    # above and the sole approved canonical-template normalization is explicit.\n    tokenizer.chat_template = template\n    require(all(tokenizer(native.apply_chat_template(row['messages'], tools=row.get('tools') or None,\n        tokenize=False, add_generation_prompt=False, enable_thinking=False), add_special_tokens=False)['input_ids']\n        == item['input_ids'] for split in rows for row,item in zip(rows[split],token_data[split])), 'loaded tokenizer token drift')\n    model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0,\n                    bias='none', target_modules=params['targets'], use_gradient_checkpointing='unsloth',\n                    random_state=3407, use_rslora=False, loftq_config=None)\n    # Custom token collator avoids TRL text-field/EOS insertion or hidden truncation.\n    def collate(batch):\n        width = max(len(item['input_ids']) for item in batch)\n        return {key:torch.tensor([item[key] + [fill]*(width-len(item[key])) for item in batch], dtype=torch.long)\n                for key,fill in (('input_ids',tokenizer.pad_token_id),('attention_mask',0),('labels',-100))}\n    class BindCheckpoint(TrainerCallback):\n        def on_save(self, args, state, control, **kwargs):\n            path = Path(args.output_dir) / ('checkpoint-' + str(state.global_step))\n            (path / 'run-contract.json').write_text(canonical(run_contract))\n            (path / 'checkpoint-hashes.json').write_text(canonical(checkpoint_hashes(path)))\n    probing = args.mode == 'probe'\n    probe_evidence = {'optimizer_attempts':0}\n    class ObserveProbe(TrainerCallback):\n        def on_pre_optimizer_step(self, args, state, control, **kwargs):\n            grads = [p.grad for p in kwargs['model'].parameters() if p.requires_grad and p.grad is not None]\n            probe_evidence.update(optimizer_attempts=probe_evidence['optimizer_attempts']+1,\n                gradient_finite=bool(grads) and all(bool(torch.isfinite(g).all()) for g in grads),\n                gradient_nonzero=any(bool(torch.any(g != 0)) for g in grads),\n                learning_rate=float(kwargs['optimizer'].param_groups[0]['lr']))\n    if probing:\n        probe_name,probe_tensor = next((name,p) for name,p in model.named_parameters() if p.requires_grad and 'lora_B' in name)\n        probe_before = probe_tensor.detach().float().cpu().clone()\n    train_data = token_data['train']\n    if probing:\n        longest = max(range(len(train_data)), key=lambda i:len(train_data[i]['input_ids']))\n        train_data = [train_data[longest]]\n    trainer = SFTTrainer(model=model, processing_class=tokenizer,\n        train_dataset=Dataset.from_list(train_data),\n        eval_dataset=None if probing else Dataset.from_list(token_data['holdout']),\n        data_collator=collate, callbacks=[ObserveProbe()] if probing else [BindCheckpoint()],\n        args=SFTConfig(output_dir=str(out / ('probe-trainer' if probing else 'trainer')),\n            per_device_train_batch_size=1, per_device_eval_batch_size=1,\n            gradient_accumulation_steps=1 if probing else 16,\n            num_train_epochs=2, max_steps=1 if probing else -1,\n            learning_rate=2e-4, warmup_ratio=0 if probing else 0.05,\n            lr_scheduler_type='constant' if probing else 'cosine',\n            optim='paged_adamw_8bit', fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),\n            logging_steps=1, save_strategy='no' if probing else 'steps', save_steps=10, save_total_limit=2,\n            eval_strategy='no' if probing else 'epoch', seed=3407, report_to='none',\n            max_length=3072, packing=False, dataset_kwargs={'skip_prepare_dataset':True},\n            remove_unused_columns=False))\n    for prepared, original in zip(trainer.train_dataset, train_data):\n        require(prepared['input_ids'] == original['input_ids'] and prepared['labels'] == original['labels'], 'trainer changed token bounds or assistant-only labels')\n    require(len(trainer.train_dataset) == len(train_data), 'trainer changed train record count')\n    if not probing:\n        require(len(trainer.eval_dataset) == len(token_data['holdout']), 'trainer changed holdout count')\n        for prepared, original in zip(trainer.eval_dataset, token_data['holdout']):\n            require(prepared['input_ids'] == original['input_ids'] and prepared['labels'] == original['labels'], 'trainer changed frozen holdout')\n    train_result = trainer.train(resume_from_checkpoint=resume_from_checkpoint)\n    if probing:\n        require(trainer.state.global_step == 1, 'probe did not complete one optimizer step')\n        skipped_update = bool(trainer.accelerator.optimizer_step_was_skipped)\n        probe_after = dict(model.named_parameters())[probe_name].detach().float().cpu()\n        longest_holdout = max(range(len(token_data['holdout'])), key=lambda i:len(token_data['holdout'][i]['input_ids']))\n        holdout_probe = trainer.evaluate(eval_dataset=Dataset.from_list([token_data['holdout'][longest_holdout]]))\n        import math\n        holdout_loss = holdout_probe.get('eval_loss')\n        probe_evidence.update(optimizer_steps=trainer.state.global_step,\n            optimizer_step_was_skipped=skipped_update,\n            loss=float(train_result.training_loss),\n            adapter_finite=all(bool(torch.isfinite(p).all()) for p in model.parameters() if p.requires_grad),\n            adapter_change_max=float((probe_after-probe_before).abs().max()),\n            longest_holdout_passed=type(holdout_loss) in (int,float) and math.isfinite(holdout_loss))\n        validate_probe_evidence(probe_evidence)\n        (out / 'probe.json').write_text(canonical(dict(probe_evidence,run_sha256=run_hash,run_contract=run_contract,\n            passed=True,longest_train_tokens=max(lengths['train']),longest_holdout_tokens=max(lengths['holdout']),\n            training_row_index=longest,holdout_row_index=longest_holdout,max_vram_bytes=torch.cuda.max_memory_allocated())))\n        # Process exit is the isolation barrier. No probe adapter is reused.\n        return\n    metrics = trainer.evaluate()\n    import math\n    require(isinstance(metrics.get('eval_loss'), (int,float)) and math.isfinite(metrics['eval_loss']), 'invalid final holdout loss')\n    (out / 'holdout-evaluation.json').write_text(canonical({'run_sha256':run_hash,'metrics':metrics,\n        'note':'Frozen final holdout loss; not a frozen repository-task A/B gate or release approval.'}))\n    model.save_pretrained(str(out / 'adapter'))\n    tokenizer.save_pretrained(str(out / 'adapter'))\n    require(shutil.disk_usage(root).free >= 20*1024**3, 'need 20 GiB free before merge/export')\n    requested = out / 'gguf'\n    sibling = requested.with_name(requested.name + '_gguf')\n    require(not requested.exists() and not sibling.exists(), 'existing export directories preserved; use a fresh runtime')\n    previous = {p.resolve() for p in out.rglob('*.gguf')}\n    export_result = model.save_pretrained_gguf(str(requested), tokenizer, quantization_method='q4_k_m')\n    require(version('gguf') == expected['gguf'], 'conversion changed the pinned GGUF reader')\n    artifacts = discover_gguf_exports(requested,out,export_result,previous)\n    quantized = []\n    for artifact in artifacts:\n        # Unsloth may leave an intermediate F16 GGUF. Only release Q4_K_M.\n        try:\n            gguf_type(artifact)  # Header selection only; never an identity certificate.\n            from gguf import GGUFReader\n            verification = validate_gguf_reader(GGUFReader(str(artifact)),artifact.stat().st_size,\n                model_config,tokenizer.get_vocab(),template,\n                {'padding':tokenizer.pad_token_id,'eos':tokenizer.eos_token_id,'bos':tokenizer.bos_token_id})\n        except (ValueError, RuntimeError, IndexError, OverflowError): continue\n        quantized.append((artifact,verification))\n    require(len(quantized) == 1, 'need exactly one fully parsed/identity-checked Q4_K_M GGUF')\n    artifact,verification = quantized[0]\n    digest = file_sha256(artifact)\n    require(artifact.stat().st_size > 1024**3, 'implausibly small 4B GGUF')\n    (out / 'artifact.json').write_text(canonical({'file':str(artifact.relative_to(root)),\n        'bytes':artifact.stat().st_size, 'sha256':digest, 'magic':'GGUF', 'verification':verification,\n        'general.file_type':15,'quantization':'Q4_K_M','run_sha256':run_hash,\n        'template_sha256':pin['template_sha256'],'base':BASE,'revision':REVISION,\n        'expected_export_identity':CANONICAL,'canonical_revision':CANONICAL_REVISION,\n        'release_ready':False,'limitations':'Conversion toolchain is provisional; independently verify serving template, smoke suite and frozen A/B gate.'}))\n    print('Export recorded; no automatic download, publication or release approval.')\n\nif __name__ == '__main__': main()\n"
(WORK / 'v2_runtime.py').write_text(RUNTIME_SOURCE, encoding='utf-8')


In [ ]:
if not globals().get('READY', False): raise RuntimeError('BLOCKED before probe')
# A subprocess owns the probe model; process exit discards its updated adapter.
subprocess.run([sys.executable, str(WORK / 'v2_runtime.py'), '--mode', 'probe'], check=True)
print((WORK / 'outputs/probe.json').read_text())


In [ ]:
if not globals().get('READY', False): raise RuntimeError('BLOCKED before training')
# Explicit checkpoint path only. Copy the entire outputs directory when persisting
# to Drive, including run-contract.json, probe.json and checkpoint contracts.
RESUME_FROM_CHECKPOINT = None
command = [sys.executable, str(WORK / 'v2_runtime.py'), '--mode', 'train']
if RESUME_FROM_CHECKPOINT: command += ['--resume', RESUME_FROM_CHECKPOINT]
subprocess.run(command, check=True)
print((WORK / 'outputs/holdout-evaluation.json').read_text())
print((WORK / 'outputs/artifact.json').read_text())


In Colab, save `outputs/` (including probe.json, checkpoints, adapter, artifact.json and the GGUF) before the runtime expires; the Files pane supports downloads, and large GGUF files may need Drive or the official CLI. No automatic account access or export download occurs.
Export is not release approval. Independently verify the serving template, 10/10 smoke suite and stock-versus-specialist frozen repository gate.
